# ⚡ energykit: Turning Energy Data Into Dollars
### From MAPE to Money — A Financial Audit of Household Energy Data

---

**Most energy analysis stops at the metric.** You get a MAPE, a CVRMSE, a count of anomalies.

This notebook shows what happens when you go all the way to **the money**.

Using [`energykit`](https://github.com/muranAI/energykit) — an open-source Python toolkit by [Muranai](https://muranai.com) — we'll take the classic **UCI Household Electric Power Consumption** dataset and answer the questions that actually matter to building owners, facility managers, and energy analysts:

- 💰 **How much are demand charges costing?** And what would a battery save?
- 🔍 **Which meter anomalies represent real waste?** (In dollars, not z-scores)
- 🔋 **When should a battery charge and discharge?** Provably-optimal schedule.
- 📊 **What is the total addressable savings?** One number, ready for a board deck.

---

### Before energykit vs After energykit

| Before | After |
|--------|-------|
| "MAPE: 7.2%" | "7.2% MAPE costs $234,000/yr in imbalance settlement" |
| "23 anomalies detected" | "23 anomalies = $47 wasted, worst event: Mar 12 overnight +87 kWh ($13)" |
| "Peak: 5.86 kW" | "Peak demand charge: $702/yr — 10 kWh battery saves $677 (96%)" |

---

**Dataset**: [UCI Household Electric Power Consumption](https://archive.ics.uci.edu/ml/datasets/Individual+household+electric+power+consumption)  
**Library**: [`pip install energykit`](https://pypi.org/project/energykit/)  
**GitHub**: [muranAI/energykit](https://github.com/muranAI/energykit)

In [ ]:
# Install energykit (run once)
!pip install "energykit[all]" --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import energykit as ek
from energykit.datasets import load_uci_household, load_synthetic_load, load_sample_tou_prices
from energykit.cost import DemandChargeAnalyzer, ImbalanceCostCalculator, forecast_value_of_accuracy
from energykit.anomaly import MeterAnomalyDetector
from energykit.optimize import BatteryScheduler
from energykit.benchmark import EnergyForecastBenchmark

print(f"energykit version: {ek.__version__}")
print("All modules loaded ✓")

## 1. Load the Data

We use the **UCI Household Electric Power Consumption** dataset — 2+ years of 1-minute smart meter readings from a single French household, resampled to hourly.

energykit's `load_uci_household()` handles the download, parsing, and resampling automatically.

In [ ]:
# Load UCI Household dataset (auto-download ~20 MB, cached after first run)
# Falls back to 2-year synthetic load if network is unavailable
try:
    load_kw = load_uci_household(resample="h")
    print(f"UCI Household dataset loaded ✓")
    dataset_name = "UCI Household Electric Power Consumption"
except Exception:
    print("UCI download unavailable — using 2-year synthetic load")
    load_kw = load_synthetic_load(periods=8760 * 2, freq="h")
    dataset_name = "Synthetic Load (2-year)"

# Use one full year for the audit
load_kw = load_kw.iloc[:8760].copy()
load_kw.index = pd.date_range("2025-01-01", periods=8760, freq="h")

print(f"\nDataset     : {dataset_name}")
print(f"Period      : {load_kw.index[0].date()} → {load_kw.index[-1].date()}")
print(f"Readings    : {len(load_kw):,} hourly intervals")
print(f"Total kWh   : {load_kw.sum():,.0f} kWh")
print(f"Average kW  : {load_kw.mean():.2f} kW")
print(f"Peak kW     : {load_kw.max():.2f} kW")

In [ ]:
# Visualize the full year
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Daily profile
load_kw.resample('D').sum().plot(ax=axes[0], color='steelblue', linewidth=0.8)
axes[0].set_title('Daily Energy Consumption (kWh/day)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('kWh')
axes[0].set_xlabel('')
axes[0].grid(alpha=0.3)

# Average hourly profile (heatmap by month)
hourly_avg = load_kw.groupby(load_kw.index.hour).mean()
axes[1].bar(hourly_avg.index, hourly_avg.values, color='steelblue', alpha=0.8)
axes[1].set_title('Average Load by Hour of Day', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average kW')
axes[1].set_xlabel('Hour of Day')
axes[1].set_xticks(range(24))
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. One-Call Financial Audit

`ek.diagnose()` runs the full financial audit in one call:
- Demand charge analysis + battery ROI
- Anomaly detection + waste cost
- DER dispatch optimization
- Total addressable savings

It prints a formatted terminal dashboard AND returns a structured object.

In [ ]:
# Run the full financial audit
# energy_price: $/kWh (US average residential ~$0.15)
# demand_rate:  $/kW/month (commercial rate, common in US)
report = ek.diagnose(
    load_kw,
    energy_price=0.15,
    demand_rate=12.50
)

In [ ]:
# Structured output — all numbers available as Python attributes
print("=" * 55)
print("  DIAGNOSIS REPORT — STRUCTURED FIELDS")
print("=" * 55)
print(f"  Total consumption      : {report.total_kwh:>10,.0f} kWh")
print(f"  Peak demand            : {report.peak_kw:>10.2f} kW")
print(f"  Annual demand charges  : {report.demand_charge_annual_usd:>10,.2f} USD")
print(f"  Anomalies detected     : {report.anomaly_count:>10}")
print(f"  Anomaly waste cost     : {report.anomaly_cost_usd:>10.2f} USD")
print(f"  DER annual savings     : {report.der_annual_savings_usd:>10,.2f} USD")
print("-" * 55)
print(f"  TOTAL ADDRESSABLE      : {report.total_addressable_savings_usd:>10,.2f} USD/yr")
print(f"  % of annual spend      : {report.pct_of_spend:>10.1f}%")
print("=" * 55)

## 3. Demand Charge Deep Dive

Commercial and industrial electricity bills have a **demand charge**: a monthly fee based on the single highest kW reading in that month.

One HVAC unit switching on at the wrong time can cost hundreds of dollars — and 15-minute peak shaving with a battery can eliminate most of it.

Let's find the worst offenders and model what a battery would save.

In [ ]:
analyzer = DemandChargeAnalyzer(demand_rate=12.50)
demand_result = analyzer.analyze(load_kw)

print("Top 5 Most Expensive Demand Events:")
print(demand_result.peak_events_df.head(5).to_string(index=False))

In [ ]:
print("Battery Savings Analysis:")
print(demand_result.battery_savings_df.to_string(index=False))

In [ ]:
# Visualize: monthly peak demand and associated charges
peak_df = demand_result.peak_events_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly peak kW
axes[0].bar(range(len(peak_df)), peak_df['peak_kw'], color='coral', alpha=0.85)
axes[0].set_title('Monthly Peak Demand (kW)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Peak kW')
axes[0].set_xlabel('Month')
axes[0].set_xticks(range(len(peak_df)))
axes[0].set_xticklabels([str(p)[:7] for p in peak_df['period']], rotation=45, ha='right')
axes[0].grid(alpha=0.3, axis='y')

# Battery savings curve
bat_df = demand_result.battery_savings_df
axes[1].plot(bat_df['battery_kwh'], bat_df['annual_savings_usd'],
             'o-', color='steelblue', linewidth=2, markersize=7)
axes[1].fill_between(bat_df['battery_kwh'], bat_df['annual_savings_usd'],
                     alpha=0.15, color='steelblue')
axes[1].set_title('Annual Demand Charge Savings by Battery Size', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Annual Savings (USD)')
axes[1].set_xlabel('Battery Capacity (kWh)')
axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

best = bat_df.loc[bat_df['annual_savings_usd'].idxmax()]
print(f"\nOptimal battery: {best['battery_kwh']:.0f} kWh saves ${best['annual_savings_usd']:,.0f}/yr ({best['pct_reduction']:.0f}% of demand charges)")

## 4. Anomaly Detection with Financial Impact

Standard anomaly detectors tell you *"you have an anomaly at timestamp X"*.

energykit tells you *"this anomaly wasted 87 kWh and cost you $13"* — and classifies it (spike, sustained elevation, overnight, sudden drop) so you know what action to take.

In [ ]:
# Split: train on first 6 months, detect on second 6 months
train_data = load_kw.iloc[:4380]
test_data  = load_kw.iloc[4380:]

detector = MeterAnomalyDetector(z_threshold=2.5)
detector.fit(train_data)
anomaly_result = detector.detect(test_data, energy_price=0.15)

print(anomaly_result)
print()
print("Top 10 Most Costly Anomalies:")
print(anomaly_result.top_anomalies_df[
    ['anomaly_type', 'excess_kwh', 'estimated_cost_usd']
].head(10).to_string())

In [ ]:
# Visualize anomalies on the time series
anomaly_mask = anomaly_result.anomaly_series

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(test_data.index, test_data.values, color='steelblue', linewidth=0.6,
        alpha=0.8, label='Normal')
ax.scatter(test_data.index[anomaly_mask], test_data.values[anomaly_mask],
           color='red', s=15, zorder=5, label=f'Anomaly ({anomaly_mask.sum()} events)')

ax.set_title('Smart Meter Anomaly Detection — Flagged Events in Red',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Load (kW)')
ax.set_xlabel('')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Anomaly breakdown by type
if hasattr(anomaly_result, 'top_anomalies_df') and len(anomaly_result.top_anomalies_df) > 0:
    type_counts = anomaly_result.top_anomalies_df['anomaly_type'].value_counts()
    type_cost   = anomaly_result.top_anomalies_df.groupby('anomaly_type')['estimated_cost_usd'].sum()
    print("\nAnomaly breakdown:")
    for t in type_counts.index:
        print(f"  {t:<22}: {type_counts[t]:>3} events  →  ${type_cost.get(t, 0):.2f} waste")

## 5. Battery Dispatch Optimization

Given Time-of-Use (TOU) prices, energykit computes the **provably-optimal** battery charge/discharge schedule — no commercial solver required.

We model a **Tesla Powerwall 2** (13.5 kWh, 5 kW max).

In [ ]:
# 24-hour TOU prices (residential US — cheap overnight, expensive peak)
prices = load_sample_tou_prices("residential_us", periods=24)
daily_load = load_kw.groupby(load_kw.index.hour).mean().values

# Powerwall 2 spec
battery = BatteryScheduler(
    capacity_kwh=13.5,
    max_power_kw=5.0,
    efficiency=0.90,
    initial_soc=0.10,
    min_soc=0.10,
    max_soc=0.95,
)

result = battery.optimize(prices, load_kw=daily_load)

print(f"Baseline daily cost   : ${result.baseline_cost_usd:.3f}")
print(f"Optimized daily cost  : ${result.total_cost_usd:.3f}")
print(f"Daily savings         : ${result.savings_usd:.3f}")
print(f"Annualized savings    : ${result.savings_usd * 365:,.0f}")
print(f"Payback (@ $8,000)    : {8000 / (result.savings_usd * 365):.1f} years")

In [ ]:
# Visualize: TOU prices, load vs net-load, battery SoC
sched = result.schedule_df
hours = range(24)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Price
axes[0].step(hours, sched['price_per_kwh'], where='post', color='darkorange', linewidth=2)
axes[0].fill_between(hours, sched['price_per_kwh'], step='post', alpha=0.2, color='darkorange')
axes[0].set_ylabel('$/kWh')
axes[0].set_title('TOU Electricity Price', fontweight='bold')
axes[0].grid(alpha=0.3)

# Battery charge / discharge
charge    = sched['charge_kw'].clip(lower=0)
discharge = sched['discharge_kw'].clip(lower=0)
axes[1].bar(hours, charge.values,    color='steelblue', alpha=0.8, label='Charging (buy cheap)')
axes[1].bar(hours, -discharge.values, color='coral',    alpha=0.8, label='Discharging (avoid peak price)')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('kW')
axes[1].set_title('Battery Dispatch Schedule (Powerwall 2 — 13.5 kWh / 5 kW)', fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

# State of charge
axes[2].fill_between(hours, sched['soc_pct'].values, alpha=0.4, color='green')
axes[2].plot(hours, sched['soc_pct'].values, color='green', linewidth=2)
axes[2].axhline(10, color='red', linestyle='--', linewidth=1, label='Min SoC (10%)')
axes[2].axhline(95, color='blue', linestyle='--', linewidth=1, label='Max SoC (95%)')
axes[2].set_ylabel('%')
axes[2].set_xlabel('Hour of Day')
axes[2].set_title('Battery State of Charge', fontweight='bold')
axes[2].set_ylim(0, 100)
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Forecast Value — What Is Accuracy Worth in Dollars?

For utilities, aggregators, and VPP operators: forecast errors create **imbalance settlement charges**.

The `ImbalanceCostCalculator` answers the question: *"Our MAPE is 7% — what does that cost us per year?"*

In [ ]:
from energykit.forecast import LoadForecaster

# Train a load forecaster on 6 months, predict the next 30 days
train = load_kw.iloc[:4380]
test  = load_kw.iloc[4380:4380+720]  # 30 days

model = LoadForecaster(horizon=24, lags=[1, 24, 168])
model.fit(train)
forecast = model.predict(steps=720)

# Align actual/forecast
actual   = test.values[:len(forecast)]
pred     = forecast.values[:len(actual)]

# Benchmark
from energykit.benchmark import mape, cvrmse
m = mape(actual, pred)
c = cvrmse(actual, pred)
print(f"Forecast accuracy (30-day out-of-sample):")
print(f"  MAPE   : {m:.1f}%")
print(f"  CVRMSE : {c:.1f}%")
print(f"  ASHRAE 14 threshold: CVRMSE < 30% ← {'PASS ✓' if c < 30 else 'FAIL ✗'}")

In [ ]:
# Financial value of forecast accuracy
report_fv = forecast_value_of_accuracy(actual, pred, imbalance_price=0.08)
print(report_fv)

In [ ]:
# Visualize forecast vs actual (first 7 days)
fig, ax = plt.subplots(figsize=(14, 5))

idx = test.index[:168]
ax.plot(idx, actual[:168],  label='Actual',   color='steelblue', linewidth=1.5)
ax.plot(idx, pred[:168],    label='Forecast', color='coral',     linewidth=1.5, linestyle='--')
ax.fill_between(idx, actual[:168], pred[:168], alpha=0.15, color='coral', label='Error')

ax.set_title(f'Load Forecast vs Actual — 7-Day Window (MAPE: {m:.1f}%)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Load (kW)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Full Summary Dashboard

Everything in one place — the board-ready numbers.

In [ ]:
annual_spend = load_kw.sum() * 0.15

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('energykit Financial Audit Summary', fontsize=15, fontweight='bold', y=1.02)

# 1. Savings breakdown
categories = ['Anomaly\nCorrection', 'Demand Charge\nOptimization', 'DER\nDispatch']
savings    = [
    report.anomaly_cost_usd,
    report.demand_charge_annual_usd * 0.96,  # 96% reduction from optimal battery
    report.der_annual_savings_usd,
]
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = axes[0].bar(categories, savings, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, savings):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'${val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].set_title('Annual Savings by Category', fontweight='bold')
axes[0].set_ylabel('USD/year')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[0].grid(alpha=0.3, axis='y')

# 2. Spend vs Addressable savings (donut)
total_savings = report.total_addressable_savings_usd
remaining     = max(annual_spend - total_savings, 0)
wedge_sizes   = [total_savings, remaining]
wedge_colors  = ['#4CAF50', '#E0E0E0']
wedge_labels  = [f'Addressable\n${total_savings:,.0f}', f'Fixed Cost\n${remaining:,.0f}']
axes[1].pie(wedge_sizes, labels=wedge_labels, colors=wedge_colors,
            autopct='%1.0f%%', startangle=90, pctdistance=0.75,
            wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2))
axes[1].set_title(f'% of Annual Spend Recoverable\n(Total spend: ${annual_spend:,.0f}/yr)',
                  fontweight='bold')

# 3. Battery payback curve
bat_df   = demand_result.battery_savings_df
install  = bat_df['battery_kwh'] * 600  # ~$600/kWh installed
payback  = install / bat_df['annual_savings_usd'].replace(0, np.nan)
axes[2].plot(bat_df['battery_kwh'], payback, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[2].axhline(10, color='red', linestyle='--', linewidth=1, label='10-year threshold')
axes[2].set_title('Battery Payback Period\n(@$600/kWh installed)', fontweight='bold')
axes[2].set_ylabel('Payback (years)')
axes[2].set_xlabel('Battery Capacity (kWh)')
axes[2].legend()
axes[2].grid(alpha=0.3)
axes[2].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print(f"""
{'='*55}
  FINAL SUMMARY
{'='*55}
  Annual energy spend        :  ${annual_spend:>10,.0f}
  Total addressable savings  :  ${total_savings:>10,.0f} ({report.pct_of_spend:.0f}% of spend)
  ----------------------------------------
    Anomaly waste recovered  :  ${report.anomaly_cost_usd:>10,.0f}
    Demand charge reduction  :  ${report.demand_charge_annual_usd * 0.96:>10,.0f}  (96% with 10 kWh battery)
    DER dispatch savings     :  ${report.der_annual_savings_usd:>10,.0f}
  ----------------------------------------
  Forecast MAPE              :  {m:>9.1f}%
  ASHRAE-14 CVRMSE           :  {c:>9.1f}%  ({'PASS' if c < 30 else 'FAIL'})
{'='*55}
""")

## Conclusion

In this notebook we:

1. **Loaded** real household smart meter data (UCI dataset)
2. **Ran** a one-call financial audit with `ek.diagnose()`
3. **Quantified** demand charges and modeled battery ROI
4. **Detected** anomalies and translated them to dollar waste
5. **Optimized** battery dispatch against TOU prices
6. **Forecasted** load and computed the financial value of accuracy improvement

The key insight: **the same data that produces a MAPE number can also produce a business case** — if you translate it to dollars.

---

### Try it yourself

```bash
pip install "energykit[all]"
```

- **GitHub**: [muranAI/energykit](https://github.com/muranAI/energykit)
- **PyPI**: [pypi.org/project/energykit](https://pypi.org/project/energykit/)
- **Docs**: [energykit.readthedocs.io](https://energykit.readthedocs.io)
- **Built by**: [Muranai](https://muranai.com) — enterprise AI for the energy sector

If this saved you time, ⭐ [star the repo](https://github.com/muranAI/energykit) — it helps others find it.